### Problem

How to create the idea of a developer organization and identify which sandbox and prod developer accounts belong to that organization.

### Notes

Merging on name has too many false positives.
Merging on email is too restrictive.

In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

In [ ]:
forbidden_emails_clause = """
                              acc.email NOT LIKE '%clover.com'
                          AND acc.email NOT LIKE '%perka.com'
                          AND acc.email NOT LIKE '%firstdata.com'
                          AND acc.email NOT LIKE '%firstdatacorp.com'
                          AND acc.email NOT LIKE '%wellsfargo.com'
                          AND acc.email NOT LIKE '%bankofamericamerchant.com'
                          AND acc.email NOT LIKE '%gmail.com'
                          AND acc.email NOT LIKE '%googlemail.com'
                          AND acc.email NOT LIKE '%yahoo.com'
                          AND acc.email NOT LIKE '%ymail.com'
                          AND acc.email NOT LIKE '%hotmail.com'
                          AND acc.email NOT LIKE '%outlook.com'
                          AND acc.email NOT LIKE '%aol.com'
                          AND acc.email NOT LIKE '%live.com'
                          AND acc.email NOT LIKE '%comcast.net'
                          AND acc.email NOT LIKE '%icloud.com'
                          AND acc.email NOT LIKE '%me.com'
                          AND acc.email NOT LIKE '%msn.com'
                          AND acc.email NOT LIKE '%sbcglobal.net'
                          AND acc.email NOT LIKE '%att.net'
                          AND acc.email NOT LIKE '%mailinator.com'
                          AND acc.email NOT LIKE '%protonmail.com'
                          """

forbidden_names_clause = """
                         d.name NOT LIKE '%test%'
                         """

q = """
    SELECT
    acc.email,
    substring(acc.email,locate("@", acc.email) + 1,char_length(acc.email)) AS domain,
    acc.name,
    d.uuid,
    d.name,
    d.approval_status
    FROM account acc
    JOIN developer d ON acc.id = d.owner_account_id
    WHERE""" + forbidden_emails_clause + """
    AND""" + forbidden_names_clause + """
    ORDER BY d.id"""

print(q)

In [ ]:
sb = SshDb("~/.clover/sb.cfg")
df_sb = df_prod = pd.read_sql(q, con=sb.conn)
sb.close()
df_sb.head()

In [ ]:
prod = Db("~/.clover/p801.cfg")
df_prod = pd.read_sql(q, con=prod.conn)
prod.close()
df_prod.head()

In [ ]:
merged_on_domain = pd.merge(left=df_prod,
                            right=df_sb,
                            suffixes=('_prod', '_sb'),
                            on='domain',
                            how='outer').sort_values(by=['domain'])


print(merged_on_domain.count())

merged_on_domain